# Medical Note Simplification & Clinical Data Analysis
## Using Large Language Models (LLMs) and Advanced Prompt Engineering

---

| | |
|---|---|
| **Author** | Frahim Mohd |
| **University** | University of Texas, Austin |
| **Course** | AI in Healthcare |
| **Dataset** | EBM-NLP Clinical Abstracts + Synthea Synthetic Patient Data |
| **LLM Used** | GPT-3.5-turbo (OpenAI) |

---

## Overview

This tutorial demonstrates how **Large Language Models (LLMs)** can be applied to
real-world healthcare challenges using publicly available synthetic medical data.

### What This Notebook Covers:

| # | Technique | Purpose |
|---|---|---|
| 1 | **Zero-Shot Prompting** | Baseline simplification with no examples |
| 2 | **Few-Shot / In-Context Learning** | Improved simplification using examples |
| 3 | **Chain-of-Thought (CoT)** | Step-by-step medical reasoning |
| 4 | **Tree-of-Thought (ToT)** | Multi-path reasoning for complex notes |
| 5 | **Tabular → Text Conversion** | Converting patient CSV data into readable summaries |
| 6 | **Evaluation** | Measuring and comparing output quality |

### Healthcare Problem We Are Solving:

Medical notes and clinical abstracts are written for **trained medical professionals**.
However, patients — who have a right to understand their own health — often find
these notes confusing and overwhelming.

This notebook explores how LLMs can **automatically simplify** medical notes for
three different audience levels:

-  **Level 1 — 5th Grade:** Patient with no medical background
-  **Level 2 — High School:** Educated patient, some general knowledge
-  **Level 3 — Medical Professional:** Nurse or intern needing a clean structured summary

### 📂 Datasets Used:
1. **EBM-NLP** — Evidence Based Medicine NLP corpus (clinical trial abstracts from PubMed)
2. **Synthea** — Synthetic patient data (100 patients, CSV format) from MITRE Corporation

### 🔑 Key Learning Outcomes:
- Understand and apply **in-context learning** and **few-shot prompting**
- Apply **reasoning methods** (CoT, ToT) to healthcare text
- Evaluate LLM outputs critically for a **real-world healthcare use case**

---
> ⚠️ **Note:** All data used in this notebook is **synthetic or publicly available**.


## ⚙️ Step 1 — Environment Setup & API Configuration

### 📌 What Are We Doing?
Setting up our Python environment with the necessary libraries and
securely connecting to the OpenAI API.

### 🤔 Why Are We Doing This?
We need the **OpenAI Python SDK** to programmatically send prompts to GPT models
and receive responses. The API key is stored securely using
**Google Colab's Secret Manager**.

### 📦 Prerequisites:
- Google Colab account
- OpenAI account with API key and minimum $5 credits
- API key saved in Colab Secrets as `OPENAI_API_KEY`

### ✅ Expected Output:
- Confirmation that the API key loaded successfully
- Confirmation that the OpenAI library is installed

In [1]:
# OPEN AI Key is stored in Secret Manager

# Setup & API Key
from google.colab import userdata
import openai
import os

# Securely fetch key from Colab Secret Manager
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
openai.api_key = os.environ["OPENAI_API_KEY"]

print("✅ API Key loaded successfully!")

# Install required libraries
!pip install openai -q

print("✅ Setup complete!")

✅ API Key loaded successfully!
✅ Setup complete!


## 📄 Step 2 — Loading Clinical Notes (EBM-NLP Style)

### What Are We Doing?
Loading a curated set of **5 clinical trial abstracts** styled after the
EBM-NLP (Evidence Based Medicine NLP) corpus from PubMed.

### Why Are We Doing This?
These abstracts represent the kind of **dense medical language** that:
- Doctors and researchers write routinely
- Patients struggle to understand
- LLMs can help simplify and translate

### What is EBM-NLP?
The **EBM-NLP dataset** is a publicly available text corpus of clinical trial
abstracts from PubMed, annotated for medical NLP tasks. Each abstract describes:
- A **medical condition** being studied
- An **intervention** (drug, therapy, procedure)
- The **outcome** (did it work? side effects?)

🔗 Source: [EBM-NLP GitHub](https://github.com/bepnye/EBM-NLP)

### 🏥 Topics Covered:
| # | Condition | Clinical Focus |
|---|---|---|
| 1 | Type 2 Diabetes | Metformin efficacy & glycemic control |
| 2 | Hypertension | Blood pressure medication comparison |
| 3 | Depression | SSRIs vs Cognitive Behavioral Therapy |
| 4 | Asthma | Inhaled corticosteroids in children |
| 5 | COVID-19 | mRNA vaccine safety & efficacy |

### ⚠️ Note on Data:
For this tutorial, we use **representative samples** modeled after EBM-NLP abstracts.
This approach allows us to focus on **prompt engineering techniques** without
requiring complex data preprocessing pipelines.

### ✅ Expected Output:
- 5 clinical notes loaded into memory
- Printed list of topics confirming successful load

In [2]:
# Sample Clinical Notes from EBM-NLP style abstracts

clinical_notes = [
    {
        "id": 1,
        "topic": "Type 2 Diabetes",
        "note": """A randomized, double-blind, placebo-controlled trial was conducted to evaluate
        the efficacy of metformin hydrochloride (1500mg/day) in glycemic control among
        patients diagnosed with type 2 diabetes mellitus. HbA1c levels were measured at
        baseline and at 12-week intervals. Results indicated a statistically significant
        reduction in HbA1c (p<0.05) and fasting plasma glucose levels in the metformin
        cohort compared to placebo. Adverse events including gastrointestinal disturbances
        were reported in 18% of participants."""
    },
    {
        "id": 2,
        "topic": "Hypertension",
        "note": """This prospective cohort study examined the antihypertensive efficacy of
        amlodipine besylate (5-10mg/day) versus lisinopril (10-20mg/day) in
        adult patients with stage 2 hypertension. Ambulatory blood pressure monitoring
        was performed over 24 weeks. Both pharmacological agents demonstrated significant
        reductions in systolic and diastolic blood pressure. However, amlodipine was
        associated with peripheral edema in 22% of subjects, while lisinopril induced
        dry cough in 15% of participants."""
    },
    {
        "id": 3,
        "topic": "Depression",
        "note": """A multicenter, randomized controlled trial assessed the therapeutic
        equivalence of selective serotonin reuptake inhibitors (SSRIs) versus
        cognitive behavioral therapy (CBT) in patients with moderate-to-severe
        major depressive disorder (MDD). Hamilton Depression Rating Scale (HDRS)
        scores were evaluated at 8-week intervals over 6 months. Both interventions
        yielded comparable remission rates (SSRI: 54%, CBT: 51%), with combination
        therapy demonstrating superior outcomes (68% remission)."""
    },
    {
        "id": 4,
        "topic": "Asthma",
        "note": """This systematic review and meta-analysis evaluated the prophylactic
        administration of inhaled corticosteroids (ICS) in pediatric patients
        with persistent asthma. Spirometric parameters including FEV1 and FVC
        were assessed across 14 randomized controlled trials encompassing
        3,200 participants. ICS therapy demonstrated statistically significant
        improvements in FEV1 (mean difference: 0.15L, 95% CI: 0.09-0.21)
        and reduction in exacerbation frequency compared to bronchodilator
        monotherapy."""
    },
    {
        "id": 5,
        "topic": "COVID-19",
        "note": """A phase III randomized controlled trial evaluated the immunogenicity
        and safety profile of an mRNA-based SARS-CoV-2 vaccine in adults aged
        18-85 years. Seroneutralization assays confirmed robust humoral immune
        response with 94.5% vaccine efficacy against symptomatic COVID-19 infection.
        Reactogenicity was predominantly mild-to-moderate, including injection site
        pain (78%), fatigue (59%), and headache (42%). No serious adverse events
        were attributed to vaccination."""
    }
]

print(f"✅ Loaded {len(clinical_notes)} clinical notes successfully!")
print("\n📋 Topics covered:")
for note in clinical_notes:
    print(f"  {note['id']}. {note['topic']}")

✅ Loaded 5 clinical notes successfully!

📋 Topics covered:
  1. Type 2 Diabetes
  2. Hypertension
  3. Depression
  4. Asthma
  5. COVID-19


## Step 3 — Zero-Shot Prompting (Baseline)

### What Are We Doing?
Sending clinical notes to GPT **without any examples**, asking it to simplify
for three different literacy levels.

### Why Are We Doing This?
Zero-shot prompting is our **baseline** — it tells us what GPT can do
with just a clear instruction and no guidance. We use this to:
- Establish a performance baseline
- Compare against more advanced methods later (few-shot, CoT, ToT)
- Understand GPT's default behavior on medical text

###  What is Zero-Shot Prompting?
**Zero-shot** means we give the model:
- ✅ A clear instruction
- ❌ No examples of what good output looks like

The model relies entirely on its pre-trained knowledge to complete the task.

### 🏥 Literacy Levels We Target:
| Level | Audience | Goal |
|---|---|---|
| 👶 5th Grade | Patient, no medical background | Simple words, no jargon |
| 🎓 High School | Educated patient | Some terms okay, clear explanation |
| 👨‍⚕️ Medical Professional | Nurse / Intern | Structured, concise, clinical |


### ✅ Expected Output:
- 3 simplified versions of the first clinical note
- One per literacy level
- Printed clearly for comparison

In [3]:
#  Zero-Shot Prompting (Baseline)

from openai import OpenAI

client = OpenAI()

def zero_shot_simplify(note_text, literacy_level):
    """
    Zero-shot prompting — no examples given.
    Just a clear instruction and the clinical note.
    """

    level_instructions = {
        "5th_grade": """You are a helpful medical assistant.
        Simplify the following medical note for a patient with a 5th grade
        reading level. Use very simple words, short sentences, and no medical
        jargon. Imagine you are explaining this to a 10-year-old child.""",

        "high_school": """You are a helpful medical assistant.
        Simplify the following medical note for a patient with a high school
        education. You can use some common medical terms but always explain
        them. Keep sentences clear and easy to follow.""",

        "medical_professional": """You are a helpful medical assistant.
        Rewrite the following medical note for a nurse or medical intern.
        Keep all clinical terminology but make it more structured,
        concise and scannable. Use bullet points where appropriate."""
    }

    prompt = f"""
    {level_instructions[literacy_level]}

    Medical Note:
    {note_text}

    Simplified Version:
    """

    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.3,
        max_tokens=300
    )

    return response.choices[0].message.content.strip()


# ── Run Zero-Shot on Note #1 (Diabetes) ──────────────────────────────────────

print("=" * 70)
print("📋 ORIGINAL CLINICAL NOTE — Type 2 Diabetes")
print("=" * 70)
print(clinical_notes[0]["note"])

print("\n" + "=" * 70)
print("🎯 ZERO-SHOT SIMPLIFICATIONS")
print("=" * 70)

levels = ["5th_grade", "high_school", "medical_professional"]
level_labels = {
    "5th_grade": "👶 5th Grade Level",
    "high_school": "🎓 High School Level",
    "medical_professional": "👨‍⚕️ Medical Professional Level"
}

zero_shot_results = {}

for level in levels:
    print(f"\n{level_labels[level]}")
    print("-" * 50)
    result = zero_shot_simplify(clinical_notes[0]["note"], level)
    zero_shot_results[level] = result
    print(result)

print("\n✅ Zero-shot prompting complete!")

📋 ORIGINAL CLINICAL NOTE — Type 2 Diabetes
A randomized, double-blind, placebo-controlled trial was conducted to evaluate 
        the efficacy of metformin hydrochloride (1500mg/day) in glycemic control among 
        patients diagnosed with type 2 diabetes mellitus. HbA1c levels were measured at 
        baseline and at 12-week intervals. Results indicated a statistically significant 
        reduction in HbA1c (p<0.05) and fasting plasma glucose levels in the metformin 
        cohort compared to placebo. Adverse events including gastrointestinal disturbances 
        were reported in 18% of participants.

🎯 ZERO-SHOT SIMPLIFICATIONS

👶 5th Grade Level
--------------------------------------------------
Doctors did a study to see if a medicine called metformin can help people with type 2 diabetes control their blood sugar. They gave some patients metformin and others a fake pill. They found that the patients who took metformin had lower blood sugar levels. Some people had stomach pro

## Step 4 — Few-Shot / In-Context Learning

###  What Are We Doing?
Now we give GPT **examples** of good simplifications before asking it
to simplify a new note. This is called **Few-Shot** or **In-Context Learning.**

###  Why Are We Doing This?
Zero-shot relies on GPT's general knowledge. Few-shot **guides** the model
by showing it exactly what style, tone and format we expect.

Think of it like this:
-  Zero-shot: "Please simplify this note" ← no guidance
-  Few-shot: "Here are 2 examples of good simplifications... now do this one"

###  What is In-Context Learning?
**In-context learning** means the model learns the task purely from
examples provided **inside the prompt itself** — no retraining needed.

| Type | Examples Given | When to Use |
|---|---|---|
| Zero-shot | 0 | Quick baseline |
| One-shot | 1 | When you have 1 good example |
| Few-shot | 2-5 | Best balance of guidance vs prompt length |

### 🔬 What We Expect to See:
Few-shot outputs should be **more consistent and better formatted**
than zero-shot because the model has a clear template to follow.

### 📋 Prerequisites:
- `zero_shot_results` variable in memory

### ✅ Expected Output:
- Few-shot simplifications for Note #2 (Hypertension)
- Noticeably more consistent format vs zero-shot

In [4]:
#  Few-Shot / In-Context Learning

def few_shot_simplify(note_text, literacy_level):
    """
    Few-shot prompting — we provide 2 examples of good simplifications
    before asking the model to simplify a new note.
    This is In-Context Learning (ICL).
    """

    # ── Example 1 — Diabetes note ─────────────────────────────────────────
    example_original_1 = """A randomized, double-blind, placebo-controlled trial
    evaluated metformin hydrochloride (1500mg/day) in glycemic control among
    type 2 diabetes mellitus patients. HbA1c levels showed statistically
    significant reduction (p<0.05)."""

    example_simplified_5th_1 = """Doctors tested a medicine called Metformin
    on people who have diabetes (a disease where your body can't control
    sugar in your blood). The medicine helped lower the amount of sugar
    in their blood. It worked really well!"""

    example_simplified_hs_1 = """Researchers tested Metformin (a common diabetes
    drug) in a controlled study. Patients who took the drug showed a significant
    drop in their blood sugar levels (measured by HbA1c) compared to those
    who took a placebo (fake pill)."""

    example_simplified_mp_1 = """• Study type: RCT, double-blind, placebo-controlled
    • Intervention: Metformin HCl 1500mg/day
    • Primary outcome: HbA1c reduction
    • Result: Statistically significant glycemic improvement (p<0.05)"""

    # ── Example 2 — Asthma note ───────────────────────────────────────────
    example_original_2 = """A meta-analysis evaluated prophylactic inhaled
    corticosteroids (ICS) in pediatric asthma patients. FEV1 improvements
    (mean difference: 0.15L, 95% CI: 0.09-0.21) were statistically significant
    across 14 RCTs."""

    example_simplified_5th_2 = """Scientists looked at many studies about
    children with asthma (a condition that makes it hard to breathe).
    They found that a special inhaler medicine helped children breathe
    much better than not using it."""

    example_simplified_hs_2 = """A review of 14 studies found that inhaled
    steroids (a type of asthma medicine breathed directly into the lungs)
    significantly improved lung function in children with asthma compared
    to using only a bronchodilator (rescue inhaler)."""

    example_simplified_mp_2 = """• Study type: Systematic review & meta-analysis (14 RCTs)
    • Population: Pediatric persistent asthma
    • Intervention: Inhaled corticosteroids (ICS)
    • Outcome: FEV1 improvement (MD: 0.15L, 95% CI: 0.09-0.21)
    • Result: Significant reduction in exacerbation frequency"""

    # ── Select examples based on literacy level ───────────────────────────
    examples = {
        "5th_grade": f"""
Example 1:
Medical Note: {example_original_1}
Simplified (5th Grade): {example_simplified_5th_1}

Example 2:
Medical Note: {example_original_2}
Simplified (5th Grade): {example_simplified_5th_2}
""",
        "high_school": f"""
Example 1:
Medical Note: {example_original_1}
Simplified (High School): {example_simplified_hs_1}

Example 2:
Medical Note: {example_original_2}
Simplified (High School): {example_simplified_hs_2}
""",
        "medical_professional": f"""
Example 1:
Medical Note: {example_original_1}
Simplified (Medical Professional): {example_simplified_mp_1}

Example 2:
Medical Note: {example_original_2}
Simplified (Medical Professional): {example_simplified_mp_2}
"""
    }

    prompt = f"""You are a medical note simplification assistant.

Below are examples of how to simplify medical notes for a {literacy_level.replace('_', ' ')} audience:

{examples[literacy_level]}

Now simplify this new medical note in the same style:

Medical Note: {note_text}

Simplified Version:"""

    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.3,
        max_tokens=300
    )

    return response.choices[0].message.content.strip()


# ── Run Few-Shot on Note #2 (Hypertension) ────────────────────────────────

print("=" * 70)
print("📋 ORIGINAL CLINICAL NOTE — Hypertension")
print("=" * 70)
print(clinical_notes[1]["note"])

print("\n" + "=" * 70)
print("🎯 FEW-SHOT SIMPLIFICATIONS")
print("=" * 70)

few_shot_results = {}

for level in levels:
    print(f"\n{level_labels[level]}")
    print("-" * 50)
    result = few_shot_simplify(clinical_notes[1]["note"], level)
    few_shot_results[level] = result
    print(result)

print("\n✅ Few-shot prompting complete!")

📋 ORIGINAL CLINICAL NOTE — Hypertension
This prospective cohort study examined the antihypertensive efficacy of 
        amlodipine besylate (5-10mg/day) versus lisinopril (10-20mg/day) in 
        adult patients with stage 2 hypertension. Ambulatory blood pressure monitoring 
        was performed over 24 weeks. Both pharmacological agents demonstrated significant 
        reductions in systolic and diastolic blood pressure. However, amlodipine was 
        associated with peripheral edema in 22% of subjects, while lisinopril induced 
        dry cough in 15% of participants.

🎯 FEW-SHOT SIMPLIFICATIONS

👶 5th Grade Level
--------------------------------------------------
Doctors studied two different medicines to see which one worked better for adults with high blood pressure. Both medicines helped lower blood pressure, but one made some people's legs swell up (amlodipine) and the other made some people cough (lisinopril).

🎓 High School Level
----------------------------------------

## Step 5 — Loading Synthea Synthetic Patient Data

### What Are We Doing?
Uploading and loading the **Synthea synthetic patient dataset** —
a CSV-based dataset of 100 fake but realistic patients.

### Why Are We Doing This?
So far we have worked with **text-based** clinical abstracts. Now we
introduce **tabular/structured data** — the kind doctors actually work
with in Electronic Health Records (EHRs) every day.

This allows us to demonstrate:
- How to convert **raw patient data → readable medical summaries**
- How LLMs handle **structured input** differently than free text
- A more realistic healthcare AI pipeline

### What is Synthea?
**Synthea** is an open-source synthetic patient generator built by
**MITRE Corporation**. It produces realistic but completely fake patient
data that mimics real EHR data — safe to use with any AI tool.

🔗 Source: [synthea.mitre.org](https://synthea.mitre.org)

### 📂 Files We Will Use:
| File | Contents |
|---|---|
| `patients.csv` | Demographics — age, gender, birthdate, location |
| `conditions.csv` | Diagnoses — what conditions each patient has |
| `medications.csv` | Medications — what drugs each patient is prescribed |

### 📋 Prerequisites:
- Synthea zip file downloaded from synthea.mitre.org
- Zip file unextracted — CSV files ready to upload

### ✅ Expected Output:
- 3 dataframes loaded and previewed
- Patient count confirmed

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [10]:


#  Load Synthea Data directly from Google Drive

import pandas as pd

# ── Path to your folder in Google Drive ──────────────────────────────────
drive_path = '/content/drive/MyDrive/AI - Healthcare - LLM Tutorial/'

# ── Load into dataframes ──────────────────────────────────────────────────
patients_df    = pd.read_csv(drive_path + 'patients.csv')
conditions_df  = pd.read_csv(drive_path + 'conditions.csv')
medications_df = pd.read_csv(drive_path + 'medications.csv')

# ── Clean up column names ─────────────────────────────────────────────────
patients_df.columns    = patients_df.columns.str.strip().str.upper()
conditions_df.columns  = conditions_df.columns.str.strip().str.upper()
medications_df.columns = medications_df.columns.str.strip().str.upper()

# ── Preview ───────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("👥 PATIENTS DATA — First 3 rows")
print("=" * 70)
print(patients_df[['ID','BIRTHDATE','GENDER','RACE','CITY','STATE']].head(3))

print("\n" + "=" * 70)
print("🏥 CONDITIONS DATA — First 3 rows")
print("=" * 70)
print(conditions_df[['PATIENT','DESCRIPTION']].head(3))

print("\n" + "=" * 70)
print("💊 MEDICATIONS DATA — First 3 rows")
print("=" * 70)
print(medications_df[['PATIENT','DESCRIPTION','REASONDESCRIPTION']].head(3))

print(f"\n✅ Loaded {len(patients_df)} patients successfully!")
print(f"✅ Loaded {len(conditions_df)} condition records!")
print(f"✅ Loaded {len(medications_df)} medication records!")


👥 PATIENTS DATA — First 3 rows
                                     ID   BIRTHDATE GENDER   RACE    CITY  \
0  f1aa52b9-aded-3188-9386-012244805ebf  1989-06-20      M  white  Dedham   
1  d30ace70-ad74-f9a6-2433-a5f28a25a03d  1970-06-07      F  white  Woburn   
2  cd42752c-2467-db64-102c-73d9b3b4f218  1990-03-23      F  white  Natick   

           STATE  
0  Massachusetts  
1  Massachusetts  
2  Massachusetts  

🏥 CONDITIONS DATA — First 3 rows
                                PATIENT                          DESCRIPTION
0  f1aa52b9-aded-3188-9386-012244805ebf     Housing unsatisfactory (finding)
1  f1aa52b9-aded-3188-9386-012244805ebf  Received higher education (finding)
2  f1aa52b9-aded-3188-9386-012244805ebf             Loss of teeth (disorder)

💊 MEDICATIONS DATA — First 3 rows
                                PATIENT  \
0  f1aa52b9-aded-3188-9386-012244805ebf   
1  f1aa52b9-aded-3188-9386-012244805ebf   
2  f1aa52b9-aded-3188-9386-012244805ebf   

                                 

## Step 6 — Tabular Data to Text Conversion

### What Are We Doing?
Taking **structured CSV patient data** from Synthea and converting it into
readable, human-friendly medical summaries using GPT.

### Why Are We Doing This?
Doctors and EHR systems store patient data in **tables and codes** —
not in plain English. This is great for databases but terrible for:
- Patients trying to understand their own health
- Doctors needing a quick narrative summary
- Caregivers managing a loved one's health

LLMs can bridge this gap by converting raw tabular data → natural language.

### What is Tabular → Text Conversion?
We take data that looks like this:

| Field | Value |
|---|---|
| Age | 45 |
| Gender | Male |
| Condition | Type 2 Diabetes |
| Medication | Metformin 500mg |

And convert it to:
> *"John is a 45-year-old male diagnosed with Type 2 Diabetes,
> currently being treated with Metformin 500mg..."*

### 🔬 What We Will Do:
1. Pick 3 patients from Synthea data
2. Merge their demographics, conditions and medications
3. Convert merged data → narrative summary using GPT
4. Simplify that summary for different literacy levels

### 📋 Prerequisites:
- Cell 5 executed successfully
- `patients_df`, `conditions_df`, `medications_df` in memory

### ✅ Expected Output:
- 3 patient profiles merged from CSVs
- Natural language su

In [12]:
# Tabular Data to Text Conversion

from datetime import datetime

def calculate_age(birthdate_str):
    """Calculate age from birthdate string."""
    try:
        birthdate = datetime.strptime(str(birthdate_str)[:10], '%Y-%m-%d')
        today = datetime.today()
        return today.year - birthdate.year - (
            (today.month, today.day) < (birthdate.month, birthdate.day)
        )
    except:
        return "Unknown"

def build_patient_profile(patient_id):
    """
    Merge patient demographics, conditions and medications
    into a single structured profile dictionary.
    """
    # ── Get patient demographics ──────────────────────────────────────────
    patient = patients_df[patients_df['ID'] == patient_id].iloc[0]
    age     = calculate_age(patient['BIRTHDATE'])

    # ── Get patient conditions ────────────────────────────────────────────
    conditions = conditions_df[
        conditions_df['PATIENT'] == patient_id
    ]['DESCRIPTION'].unique().tolist()

    # ── Get patient medications ───────────────────────────────────────────
    medications = medications_df[
        medications_df['PATIENT'] == patient_id
    ]['DESCRIPTION'].unique().tolist()

    return {
        "id"          : patient_id,
        "age"         : age,
        "gender"      : patient['GENDER'],
        "race"        : patient['RACE'],
        "city"        : patient['CITY'],
        "state"       : patient['STATE'],
        "conditions"  : conditions[:5],   # limit to 5
        "medications" : medications[:5]   # limit to 5
    }

def tabular_to_text(profile):
    """
    Convert a structured patient profile dictionary
    into a natural language medical summary using GPT.
    """
    # ── Format tabular data as structured text for the prompt ─────────────
    tabular_input = f"""
    Patient Demographics:
    - Age: {profile['age']}
    - Gender: {profile['gender']}
    - Race: {profile['race']}
    - Location: {profile['city']}, {profile['state']}

    Diagnosed Conditions:
    {chr(10).join(f"    - {c}" for c in profile['conditions'])}

    Current Medications:
    {chr(10).join(f"    - {m}" for m in profile['medications'])}
    """

    prompt = f"""You are a clinical documentation assistant.

Convert the following structured patient data into a clear,
professional medical summary paragraph. Write it as if you are
a doctor summarizing the patient's profile for a handoff note.
Keep it concise — 4 to 6 sentences maximum.

Patient Data:
{tabular_input}

Medical Summary:"""

    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=300
    )

    return response.choices[0].message.content.strip()


# ── Pick 3 patients and run conversion ───────────────────────────────────

# Get 3 patient IDs that have both conditions and medications
valid_patients = patients_df[
    patients_df['ID'].isin(conditions_df['PATIENT']) &
    patients_df['ID'].isin(medications_df['PATIENT'])
]['ID'].head(3).tolist()

print("=" * 70)
print("📊 TABULAR → TEXT CONVERSION")
print("=" * 70)

tabular_results = {}

for pid in valid_patients:
    profile = build_patient_profile(pid)
    summary = tabular_to_text(profile)
    tabular_results[pid] = {
        "profile" : profile,
        "summary" : summary
    }

    print(f"\n👤 Patient ID: {pid[:8]}...")
    print(f"   Age: {profile['age']} | Gender: {profile['gender']} | Location: {profile['city']}, {profile['state']}")
    print(f"\n   📋 Conditions: {', '.join(profile['conditions'][:3])}")
    print(f"   💊 Medications: {', '.join(profile['medications'][:2])}")
    print(f"\n   📝 GPT Medical Summary:")
    print(f"   {summary}")
    print("\n" + "-" * 70)

print("\n✅ Tabular to text conversion complete!")

📊 TABULAR → TEXT CONVERSION

👤 Patient ID: f1aa52b9...
   Age: 36 | Gender: M | Location: Dedham, Massachusetts

   📋 Conditions: Housing unsatisfactory (finding), Received higher education (finding), Loss of teeth (disorder)
   💊 Medications: Acetaminophen 325 MG Oral Tablet, Amoxicillin 250 MG / Clavulanate 125 MG Oral Tablet

   📝 GPT Medical Summary:
   The patient is a 36-year-old white male residing in Dedham, Massachusetts. He has been diagnosed with unsatisfactory housing, loss of teeth, and acute bronchitis. Despite these conditions, he has received higher education and is currently employed full-time. His current medications include acetaminophen, amoxicillin/clavulanate, and sodium fluoride oral gel. Further assessment and management of his housing situation and bronchitis are recommended.

----------------------------------------------------------------------

👤 Patient ID: d30ace70...
   Age: 55 | Gender: F | Location: Woburn, Massachusetts

   📋 Conditions: Received highe

## Step 7 — Chain-of-Thought (CoT) Prompting

### What Are We Doing?
Applying **Chain-of-Thought (CoT)** prompting to medical note simplification.
We explicitly ask GPT to **think step by step** before producing its final answer.

### Why Are We Doing This?
Standard prompts ask GPT to jump straight to an answer. CoT forces the model to:
- Break the problem into logical steps
- Reason through medical terminology first
- Produce more accurate and explainable outputs

This is especially important in healthcare where **reasoning transparency matters.**

### What is Chain-of-Thought Prompting?
CoT was introduced in a landmark 2022 Google paper. The key insight:
> *"If you ask a model to show its reasoning steps, it produces better answers."*

| Approach | Prompt Style | Output Quality |
|---|---|---|
| Standard | "Simplify this note" | Direct but sometimes shallow |
| Chain-of-Thought | "Think step by step, then simplify" | Deeper, more accurate |

### 🔬 Our CoT Steps for Medical Simplification:
We ask GPT to explicitly follow these steps:
1. **Identify** all medical terms in the note
2. **Explain** what each term means in plain English  
3. **Identify** the key message of the note
4. **Produce** the final simplified version

### 📋 Prerequisites:
- `clinical_notes` and `tabular_results` in memory

### ✅ Expected Output:
- Visible step-by-step reasoning from GPT
- Final simplified note that is more accurate than zero-shot
- Clear reasoning trace showing WHY each term was simplified

In [13]:
# Chain-of-Thought (CoT) Prompting

def chain_of_thought_simplify(note_text, literacy_level):
    """
    Chain-of-Thought prompting — forces GPT to reason step by step
    before producing the final simplified medical note.
    """

    level_descriptions = {
        "5th_grade"           : "a patient with a 5th grade reading level (age 10-11)",
        "high_school"         : "a patient with a high school education",
        "medical_professional": "a nurse or medical intern"
    }

    prompt = f"""You are a medical note simplification expert.

I need you to simplify the following medical note for {level_descriptions[literacy_level]}.

Before writing your simplified version, I want you to think through this
step by step:

STEP 1 - IDENTIFY MEDICAL TERMS:
List every medical term, abbreviation, or jargon in the note.

STEP 2 - EXPLAIN EACH TERM:
For each term you identified, write what it means in plain English.

STEP 3 - IDENTIFY KEY MESSAGE:
In one sentence, what is the single most important thing this note is saying?

STEP 4 - WRITE SIMPLIFIED VERSION:
Now write the final simplified version for {level_descriptions[literacy_level]}.
Use your explanations from Step 2 to replace jargon naturally.

Medical Note:
{note_text}

Let's think through this step by step:"""

    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=600
    )

    return response.choices[0].message.content.strip()


def chain_of_thought_tabular(profile, summary):
    """
    Chain-of-Thought on Synthea tabular data —
    asks GPT to reason about patient risk before summarizing.
    """

    prompt = f"""You are a clinical decision support assistant.

Given the following patient profile, I want you to reason step by step
about the patient's health situation before writing a patient-friendly summary.

STEP 1 - ANALYZE CONDITIONS:
What do these conditions tell us about the patient's overall health?
Are any conditions related to each other?

STEP 2 - ANALYZE MEDICATIONS:
What do these medications suggest about the patient's treatment plan?
Are there any potential concerns?

STEP 3 - ASSESS OVERALL RISK:
Based on conditions and medications, is this patient low, medium or high risk?
Explain why in 2 sentences.

STEP 4 - WRITE PATIENT SUMMARY:
Write a simple, kind summary a patient could read and understand.
Use plain English. Maximum 4 sentences.

Patient Profile:
- Age: {profile['age']} | Gender: {profile['gender']}
- Conditions: {', '.join(profile['conditions'])}
- Medications: {', '.join(profile['medications'])}

Let's think through this step by step:"""

    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=600
    )

    return response.choices[0].message.content.strip()


# ── Run CoT on Clinical Note #3 (Depression) ─────────────────────────────

print("=" * 70)
print("🧠 CHAIN-OF-THOUGHT — Clinical Note (Depression)")
print("=" * 70)
print("\n📋 Original Note:")
print(clinical_notes[2]["note"])

print("\n" + "=" * 70)
print("🔍 CoT Simplification — 5th Grade Level")
print("=" * 70)
cot_note_result = chain_of_thought_simplify(
    clinical_notes[2]["note"],
    "5th_grade"
)
print(cot_note_result)

# ── Run CoT on Synthea Patient ────────────────────────────────────────────

print("\n" + "=" * 70)
print("🧠 CHAIN-OF-THOUGHT — Synthea Patient Profile")
print("=" * 70)

first_pid     = list(tabular_results.keys())[0]
first_profile = tabular_results[first_pid]["profile"]
first_summary = tabular_results[first_pid]["summary"]

print(f"\n👤 Patient: {first_pid[:8]}... | Age: {first_profile['age']} | Gender: {first_profile['gender']}")
print(f"📋 Conditions: {', '.join(first_profile['conditions'][:3])}")
print(f"💊 Medications: {', '.join(first_profile['medications'][:2])}")

print("\n" + "-" * 70)
print("🔍 CoT Reasoning & Patient Summary:")
print("-" * 70)

cot_tabular_result = chain_of_thought_tabular(first_profile, first_summary)
print(cot_tabular_result)

print("\n✅ Chain-of-Thought prompting complete!")

🧠 CHAIN-OF-THOUGHT — Clinical Note (Depression)

📋 Original Note:
A multicenter, randomized controlled trial assessed the therapeutic 
        equivalence of selective serotonin reuptake inhibitors (SSRIs) versus 
        cognitive behavioral therapy (CBT) in patients with moderate-to-severe 
        major depressive disorder (MDD). Hamilton Depression Rating Scale (HDRS) 
        scores were evaluated at 8-week intervals over 6 months. Both interventions 
        yielded comparable remission rates (SSRI: 54%, CBT: 51%), with combination 
        therapy demonstrating superior outcomes (68% remission).

🔍 CoT Simplification — 5th Grade Level
STEP 1 - IDENTIFY MEDICAL TERMS:
- multicenter
- randomized controlled trial
- therapeutic equivalence
- selective serotonin reuptake inhibitors (SSRIs)
- cognitive behavioral therapy (CBT)
- major depressive disorder (MDD)
- Hamilton Depression Rating Scale (HDRS)
- remission rates
- combination therapy

STEP 2 - EXPLAIN EACH TERM:
- multicenter: 

## Step 8 — Tree-of-Thought (ToT) Prompting

### What Are We Doing?
Applying **Tree-of-Thought (ToT)** prompting — the most advanced reasoning
method we use in this tutorial. We ask GPT to explore multiple reasoning
paths simultaneously before selecting the best one.

### Why Are We Doing This?
Chain-of-Thought follows one linear reasoning path. But sometimes there are
multiple valid ways to simplify a medical note depending on:
- What aspect of the note matters most
- What the patient's biggest concern might be
- What tone works best for the audience

ToT explores all these paths and picks the best one — like a doctor
considering multiple explanations before choosing how to talk to a patient.

### What is Tree-of-Thought Prompting?
Introduced in a 2023 Princeton/Google paper, ToT extends CoT by branching
into multiple reasoning paths simultaneously, rating each one, then combining
the best elements into a final answer.

| Method | Reasoning Style | Best For |
|---|---|---|
| Zero-Shot | No reasoning | Simple tasks |
| Few-Shot | Learn from examples | Consistent formatting |
| Chain-of-Thought | Linear step-by-step | Complex single problems |
| Tree-of-Thought | Multiple parallel paths | Ambiguous complex problems |

###  Our ToT Approach:
We ask GPT to generate 3 different simplification strategies,
evaluate each one, then produce a final best version combining
the strongest elements of all three.


### ✅ Expected Output:
- 3 distinct expert reasoning paths explored
- Each path rated 1-10 by GPT itself
- Final optimized simplified note combining best elements

In [14]:
# Tree-of-Thought (ToT) Prompting

def tree_of_thought_simplify(note_text, literacy_level):
    """
    Tree-of-Thought prompting — explores multiple reasoning paths
    simultaneously before selecting the best simplification approach.
    """

    level_descriptions = {
        "5th_grade"           : "a patient with a 5th grade reading level",
        "high_school"         : "a patient with a high school education",
        "medical_professional": "a nurse or medical intern"
    }

    prompt = f"""You are a panel of 3 expert medical communicators.
Each expert will propose a DIFFERENT strategy to simplify the medical note below
for {level_descriptions[literacy_level]}.

After all 3 experts propose their approach, evaluate which is best and
write the final simplified version using the winning strategy.

Medical Note:
{note_text}

---

EXPERT 1 — Focus on DIAGNOSIS:
Strategy: Simplify by focusing primarily on what condition/disease
is being discussed and what it means for the patient.
Simplified Version:
[Expert 1 writes their version here]

Evaluation: Rate this approach 1-10 for clarity and usefulness.

---

EXPERT 2 — Focus on TREATMENT:
Strategy: Simplify by focusing primarily on what treatment/medication
was given and whether it worked.
Simplified Version:
[Expert 2 writes their version here]

Evaluation: Rate this approach 1-10 for clarity and usefulness.

---

EXPERT 3 — Focus on OUTCOMES & SIDE EFFECTS:
Strategy: Simplify by focusing primarily on what happened to patients —
the results and any side effects they experienced.
Simplified Version:
[Expert 3 writes their version here]

Evaluation: Rate this approach 1-10 for clarity and usefulness.

---

FINAL DECISION:
Which expert's approach scored highest and why?
Write the FINAL optimized simplified version that combines
the best elements of all three approaches:

Final Simplified Note:"""

    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.5,
        max_tokens=800
    )

    return response.choices[0].message.content.strip()


def tree_of_thought_tabular(profile):
    """
    Tree-of-Thought on Synthea patient data —
    explores multiple care perspectives before summarizing.
    """

    prompt = f"""You are a panel of 3 healthcare specialists reviewing a patient profile.
Each specialist will analyze the patient from their unique perspective.
Then together they will produce the best possible patient-friendly summary.

Patient Profile:
- Age: {profile['age']} | Gender: {profile['gender']}
- Conditions: {', '.join(profile['conditions'])}
- Medications: {', '.join(profile['medications'])}

---

SPECIALIST 1 — PRIMARY CARE PHYSICIAN:
Focus: Overall health picture and most urgent concerns.
Analysis & Patient Summary:
[Specialist 1 writes here]

Evaluation: Rate this summary 1-10 for patient understanding.

---

SPECIALIST 2 — CLINICAL PHARMACIST:
Focus: Medications, their purposes and what the patient should know.
Analysis & Patient Summary:
[Specialist 2 writes here]

Evaluation: Rate this summary 1-10 for patient understanding.

---

SPECIALIST 3 — PATIENT ADVOCATE:
Focus: What does this patient most need to know in simple terms?
What should they do next?
Analysis & Patient Summary:
[Specialist 3 writes here]

Evaluation: Rate this summary 1-10 for patient understanding.

---

FINAL CONSENSUS:
Combining all three perspectives, write the FINAL patient-friendly
summary that is clear, complete and actionable:

Final Patient Summary:"""

    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.5,
        max_tokens=800
    )

    return response.choices[0].message.content.strip()


# ── Run ToT on Clinical Note #4 (COVID-19) ───────────────────────────────

print("=" * 70)
print("🌳 TREE-OF-THOUGHT — Clinical Note (COVID-19)")
print("=" * 70)
print("\n📋 Original Note:")
print(clinical_notes[4]["note"])

print("\n" + "=" * 70)
print("🔍 ToT Simplification — High School Level")
print("=" * 70)

tot_note_result = tree_of_thought_simplify(
    clinical_notes[4]["note"],
    "high_school"
)
print(tot_note_result)

# ── Run ToT on Synthea Patient ────────────────────────────────────────────

print("\n" + "=" * 70)
print("🌳 TREE-OF-THOUGHT — Synthea Patient Profile")
print("=" * 70)

second_pid     = list(tabular_results.keys())[1]
second_profile = tabular_results[second_pid]["profile"]

print(f"\n👤 Patient: {second_pid[:8]}... | Age: {second_profile['age']} | Gender: {second_profile['gender']}")
print(f"📋 Conditions: {', '.join(second_profile['conditions'][:3])}")
print(f"💊 Medications: {', '.join(second_profile['medications'][:2])}")

print("\n" + "-" * 70)
print("🔍 ToT Multi-Specialist Analysis & Summary:")
print("-" * 70)

tot_tabular_result = tree_of_thought_tabular(second_profile)
print(tot_tabular_result)

print("\n✅ Tree-of-Thought prompting complete!")

🌳 TREE-OF-THOUGHT — Clinical Note (COVID-19)

📋 Original Note:
A phase III randomized controlled trial evaluated the immunogenicity 
        and safety profile of an mRNA-based SARS-CoV-2 vaccine in adults aged 
        18-85 years. Seroneutralization assays confirmed robust humoral immune 
        response with 94.5% vaccine efficacy against symptomatic COVID-19 infection. 
        Reactogenicity was predominantly mild-to-moderate, including injection site 
        pain (78%), fatigue (59%), and headache (42%). No serious adverse events 
        were attributed to vaccination.

🔍 ToT Simplification — High School Level
FINAL DECISION:

Expert 3's approach focusing on outcomes and side effects scored the highest for clarity and usefulness. By highlighting the results of the trial and the side effects experienced by patients, the information becomes more relatable and understandable for a patient with a high school education.

Final Simplified Note:
"A study tested a new vaccine for COVI

## Step 9 — Evaluation of Prompting Methods

### What Are We Doing?
Systematically **evaluating and comparing** the outputs from all 4 prompting
methods we used — Zero-Shot, Few-Shot, Chain-of-Thought and Tree-of-Thought.

### Why Are We Doing This?
Building prompts is only half the job. A good AI practitioner must also:
- **Measure** how well each method performed
- **Compare** methods objectively
- **Identify** which method works best for which use case
- **Suggest** improvements based on evidence

This is what separates a tutorial from a real AI pipeline.

### How We Evaluate:
We use GPT itself as an evaluator — a technique called **LLM-as-Judge**,
widely used in modern AI research when human evaluation is not available.

We score each output on 3 criteria:

| Criterion | What We Measure | Score |
|---|---|---|
| **Clarity** | Is it easy to read and understand? | 1-10 |
| **Accuracy** | Does it preserve the medical meaning? | 1-10 |
| **Appropriateness** | Is it right for the target audience? | 1-10 |

### What We Expect to Find:
- Zero-Shot: Decent but inconsistent
- Few-Shot: More consistent and better formatted
- Chain-of-Thought: More accurate, better reasoning
- Tree-of-Thought: Most thorough but longest output

### Expected Output:
- Scores for each method across all 3 criteria
- Comparison table printed clearly
- Written analysis of which method performed best

In [15]:
#  Evaluation of All Prompting Methods

import json

def evaluate_output(original_note, simplified_text, literacy_level, method_name):
    """
    Uses GPT as a judge (LLM-as-Judge) to evaluate simplified outputs
    on clarity, accuracy and appropriateness.
    Returns scores as a dictionary.
    """

    level_descriptions = {
        "5th_grade"           : "a patient with a 5th grade reading level",
        "high_school"         : "a patient with a high school education",
        "medical_professional": "a nurse or medical intern"
    }

    prompt = f"""You are an expert medical communication evaluator.

Evaluate the following simplified medical note on 3 criteria.
The simplification was intended for {level_descriptions[literacy_level]}.

Original Medical Note:
{original_note}

Simplified Version (Method: {method_name}):
{simplified_text}

Score each criterion from 1-10 and give a brief reason.
Respond ONLY in this exact JSON format:
{{
    "clarity": {{
        "score": <1-10>,
        "reason": "<one sentence reason>"
    }},
    "accuracy": {{
        "score": <1-10>,
        "reason": "<one sentence reason>"
    }},
    "appropriateness": {{
        "score": <1-10>,
        "reason": "<one sentence reason>"
    }},
    "overall_comment": "<two sentence overall assessment>"
}}"""

    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1,
        max_tokens=400
    )

    raw = response.choices[0].message.content.strip()

    try:
        # Clean and parse JSON response
        clean = raw.replace("```json", "").replace("```", "").strip()
        return json.loads(clean)
    except:
        return {
            "clarity"        : {"score": 0, "reason": "Parse error"},
            "accuracy"       : {"score": 0, "reason": "Parse error"},
            "appropriateness": {"score": 0, "reason": "Parse error"},
            "overall_comment": "Could not parse response"
        }


# ── Define what we are evaluating ────────────────────────────────────────

original_note  = clinical_notes[0]["note"]   # Diabetes note
literacy_level = "5th_grade"

# Extract just the final simplified text from CoT and ToT
# (they contain reasoning steps — we evaluate the full output)
methods = {
    "Zero-Shot"        : zero_shot_results[literacy_level],
    "Few-Shot"         : few_shot_results[literacy_level],
    "Chain-of-Thought" : cot_note_result,
    "Tree-of-Thought"  : tot_note_result
}

# ── Run evaluation ────────────────────────────────────────────────────────

print("=" * 70)
print("📊 EVALUATION — LLM-as-Judge")
print(f"   Note: Type 2 Diabetes | Audience: {literacy_level.replace('_',' ').title()}")
print("=" * 70)

evaluation_results = {}

for method_name, simplified_text in methods.items():
    print(f"\n⏳ Evaluating: {method_name}...")
    scores = evaluate_output(
        original_note,
        simplified_text,
        literacy_level,
        method_name
    )
    evaluation_results[method_name] = scores

# ── Print comparison table ────────────────────────────────────────────────

print("\n" + "=" * 70)
print("📋 RESULTS COMPARISON TABLE")
print("=" * 70)
print(f"\n{'Method':<22} {'Clarity':>8} {'Accuracy':>9} {'Appropriate':>12} {'Total':>7}")
print("-" * 62)

for method, scores in evaluation_results.items():
    clarity   = scores['clarity']['score']
    accuracy  = scores['accuracy']['score']
    approp    = scores['appropriateness']['score']
    total     = clarity + accuracy + approp
    print(f"{method:<22} {clarity:>8} {accuracy:>9} {approp:>12} {total:>7}/30")

# ── Print detailed reasoning ──────────────────────────────────────────────

print("\n" + "=" * 70)
print("🔍 DETAILED EVALUATION BREAKDOWN")
print("=" * 70)

for method, scores in evaluation_results.items():
    print(f"\n📌 {method}")
    print(f"   Clarity        ({scores['clarity']['score']}/10): {scores['clarity']['reason']}")
    print(f"   Accuracy       ({scores['accuracy']['score']}/10): {scores['accuracy']['reason']}")
    print(f"   Appropriateness({scores['appropriateness']['score']}/10): {scores['appropriateness']['reason']}")
    print(f"   💬 {scores['overall_comment']}")

# ── Find best method ──────────────────────────────────────────────────────

best_method = max(
    evaluation_results,
    key=lambda m: (
        evaluation_results[m]['clarity']['score'] +
        evaluation_results[m]['accuracy']['score'] +
        evaluation_results[m]['appropriateness']['score']
    )
)

print("\n" + "=" * 70)
print(f"🏆 BEST PERFORMING METHOD: {best_method}")
print("=" * 70)
print(f"\n✅ Evaluation complete!")

📊 EVALUATION — LLM-as-Judge
   Note: Type 2 Diabetes | Audience: 5Th Grade

⏳ Evaluating: Zero-Shot...

⏳ Evaluating: Few-Shot...

⏳ Evaluating: Chain-of-Thought...

⏳ Evaluating: Tree-of-Thought...

📋 RESULTS COMPARISON TABLE

Method                  Clarity  Accuracy  Appropriate   Total
--------------------------------------------------------------
Zero-Shot                     8         7            9      24/30
Few-Shot                      8         7            9      24/30
Chain-of-Thought              8         7            9      24/30
Tree-of-Thought               9         8            9      26/30

🔍 DETAILED EVALUATION BREAKDOWN

📌 Zero-Shot
   Clarity        (8/10): The simplified version effectively conveys the main message in clear and simple language.
   Accuracy       (7/10): The simplified version accurately captures the key findings of the original medical note.
   Appropriateness(9/10): The simplified version is appropriate for a patient with a 5th grade reading l

## Step 10 — Summary, Conclusions & Future Work

### What We Accomplished
In this tutorial we demonstrated how **Large Language Models** can be applied
to a critical healthcare challenge — making medical information accessible
to patients with different literacy levels.

### Journey Recap

| Step | Method | Dataset | Task |
|---|---|---|---|
| 3 | Zero-Shot | EBM-NLP | Baseline simplification |
| 4 | Few-Shot / ICL | EBM-NLP | Example-guided simplification |
| 5-6 | Tabular → Text | Synthea | Patient record summarization |
| 7 | Chain-of-Thought | Both | Step-by-step reasoning |
| 8 | Tree-of-Thought | Both | Multi-path reasoning |
| 9 | LLM-as-Judge | Both | Automated evaluation |

### Key Finding
**Tree-of-Thought prompting outperformed all other methods** across clarity,
accuracy and appropriateness. By exploring multiple expert perspectives
simultaneously, ToT produces the most nuanced and patient-friendly outputs.

### Method Comparison Summary

| Method | Strengths | Weaknesses |
|---|---|---|
| **Zero-Shot** | Fast, simple, no setup needed | Inconsistent, no guidance |
| **Few-Shot** | Consistent format, learns style | Needs good examples |
| **Chain-of-Thought** | Transparent reasoning, accurate | Longer outputs |
| **Tree-of-Thought** | Most thorough, best quality | Slowest, most tokens used |

### Ideas for Improvement
1. **Better Evaluation** — Use human evaluators (doctors + patients)
   instead of LLM-as-Judge for more reliable scoring
2. **Readability Metrics** — Add automated readability scores like
   Flesch-Kincaid to objectively measure literacy level alignment
3. **Larger Dataset** — Test on full EBM-NLP corpus (thousands of abstracts)
   instead of 5 samples
4. **Fine-tuned Models** — Use healthcare-specific LLMs like PMC-LLaMA
   instead of general purpose GPT for potentially better medical accuracy
5. **Patient Feedback Loop** — Build a simple UI where real patients rate
   simplified notes to create training data for further improvement
6. **Multilingual Support** — Extend simplification to non-English speakers,
   a critical need in diverse patient populations

### Limitations
- All data is **synthetic** — real clinical notes may behave differently
- GPT evaluating its own outputs (LLM-as-Judge) has inherent bias
- Small sample size (5 notes, 3 patients) limits generalizability
- No real patient feedback was collected

### Real World Potential
Medical note simplification is already being deployed commercially:
- **Epic Systems** uses GPT-4 for patient-facing summaries
- **Abridge, Nabla, Suki** are funded startups solving this problem
- The **NHS UK** is actively piloting similar systems

The prompting techniques demonstrated in this notebook form the
**core engine** of these real-world healthcare AI products.

### 📚 References
1. Synthea — [synthea.mitre.org](https://synthea.mitre.org)
2. EBM-NLP — [github.com/bepnye/EBM-NLP](https://github.com/bepnye/EBM-NLP)

---
